# [실습] LangGraph Memory (하네스의 영속성 계층)

LangGraph의 메모리 시스템은 두 가지 계층으로 구성됩니다.

| 메모리 유형 | 저장 범위 | 구현 | 활용 예 |
|-----------|---------|-----|--------|
| 단기 메모리 (Checkpointer) | Thread 내 | `SqliteSaver` | 대화 히스토리 |
| 장기 메모리 (Store) | Cross-thread | `InMemoryStore` | 사용자 프로필, 선호도 |

> 하네스 엔지니어링 관점: 메모리는 에이전트 하네스의 영속성 계층입니다.
> 에이전트가 프로세스 재시작 후에도 맥락을 유지하고, 사용자별 개인화된 경험을 제공하려면
> 견고한 메모리 시스템이 필수입니다.
>
> ```
> Harness 영속성 계층:
> - 단기 메모리: SqliteSaver (Thread 내 대화 복원)
> - 장기 메모리: InMemoryStore (Cross-thread 정보 공유)
> ```

## 학습 목표

1. `SqliteSaver` Checkpointer로 영구 저장 가능한 대화 메모리 구현
2. `thread_id`를 활용한 멀티 세션 관리
3. 프로세스 재시작 후에도 대화 복원 확인
4. `InMemoryStore`로 Cross-thread 장기 메모리 구현
5. `ToolRuntime`으로 도구에서 Store 접근
6. 임베딩 기반 유사도 검색으로 관련 메모리 조회

> 하네스의 영속성 계층이 없으면, 에이전트는 비효율적으로 매번 새로운 정보를 주입받아야 합니다.

In [1]:
%pip install langgraph-checkpoint-sqlite langchain langchain-openai langchain-community -q

Note: you may need to restart the kernel to use updated packages.


## 1. 환경 설정

In [2]:
import os
import uuid
import sqlite3
from dotenv import load_dotenv

load_dotenv('.env', override=True)

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.tools import ToolRuntime
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.store.memory import InMemoryStore

from select_model import load_model
llm = load_model(platform='vllm', enable_thinking=True, max_tokens=16384, temperature=1.0, top_p=0.95, presence_penalty=1.5, extra_body={"top_k": 20, "min_p": 0.0, "repetition_penalty": 1.0})

print('환경 설정 완료')

환경 설정 완료


---
## 2. 단기 메모리: Checkpointer

### InMemorySaver vs SqliteSaver

| 항목 | `InMemorySaver` | `SqliteSaver` |
|------|----------------|---------------|
| 저장 위치 | 메모리 (RAM) | SQLite DB 파일 |
| 프로세스 종료 시 | 소멸 | 영구 보존 |
| 용도 | 개발/테스트 | 운영 환경 |
| 설정 | 없음 | DB 경로 지정 |

> 마이그레이션 노트: `MemorySaver`는 deprecated되었습니다.
> 개발 시에도 `SqliteSaver`를 사용하면 디버깅이 편리합니다.

In [4]:
# SQLite DB 파일에 체크포인트 저장
import uuid

db_id = str(uuid.uuid4())[:8]
# 랜덤 문자열 8글자 (반복 실행시 중복 방지)

DB_PATH = f"memory_lab_{db_id}.db"
conn = sqlite3.connect(DB_PATH, check_same_thread=False)
checkpointer = SqliteSaver(conn)

agent = create_agent(
    llm,
    tools=[],
    system_prompt="당신은 유능한 AI 어시스턴트입니다. 한국어로 간략하게 답변하세요.",
    checkpointer=checkpointer,
)

print(f'SqliteSaver 에이전트 생성 완료 (DB: {DB_PATH})')

SqliteSaver 에이전트 생성 완료 (DB: memory_lab_ce36e2b9.db)


In [5]:
# Thread 1: 대화 메모리 테스트
config_t1 = {"configurable": {"thread_id": "thread-1"}}

r1 = agent.invoke(
    {"messages": [{"role": "user", "content": "나는 Context Engineering에 관심이 많다."}]},
    config=config_t1
)
print(f'턴 1: {r1["messages"][-1].text}')

턴 1: 

Context Engineering은 AI(특히 대형 언어 모델)가 더 정확하고 일관된 결과를 생산하도록 `컨텍스트(정보·문맥·제약·도메인 지식 등)`를 구조화·관리·최적화하는 분야입니다. 프롬프트 엔지니어링을 넘어 RAG, 컨텍스트 윈도우 효율화, 상태/메모리 설계, 도메인별 지식 파이프라인 구축 등을 포함합니다.

구체적으로 어떤 부분에 관심이 있으신가요? (예: 핵심 기법, 관련 프레임워크/도구, 실제 적용 사례, 학습 경로 등) 알려주시면 바로 정리해 드립니다.


In [6]:
# 두 번째 턴 (같은 thread_id)
r2 = agent.invoke(
    {"messages": [{"role": "user", "content": "내 관심사에 대해 알려줘."}]},
    config=config_t1
)
print(f'턴 2 (같은 스레드): {r2["messages"][-1].text}')

턴 2 (같은 스레드): 

현재까지 공유하신 정보로는 **'컨텍스트 엔지니어링'**이 가장 명확한 관심사입니다. 이 분야를 깊이 다룰 경우 자연스럽게 다음 하위 영역에 주목하게 됩니다:

• **RAG & 컨텍스트 최적화**: 청크 분할 전략, 벡터 검색 연계, 컨텍스트 윈도우 효율적 활용  
• **메모리·상태 관리**: 단기/장기 기억 구조, 세션/멀티턴 컨텍스트 갱신 로직  
• **도메인 지식 파이프라인**: 정형/비정형 데이터 → 검증된 컨텍스트 템플릿 자동화  
• **프롬프트 구조화**: 역할·제약·출력 형식 명시화, 컨텍스트 우선순위·필터링 기법  

구체적인 초점(기술 구현, 아키텍처 설계, 최신 연구, 실무 적용 등)이나 목표 수준을 알려주시면 바로 맞춤형 참고자료·프레임워크·실행 로드맵으로 정리해 드립니다.


In [7]:
# 다른 스레드: 앞선 대화 기억 못함
config_t2 = {"configurable": {"thread_id": "thread-2"}}

r3 = agent.invoke(
    {"messages": [{"role": "user", "content": "나에 대해 아는 것을 전부 알려줘"}]},
    config=config_t2
)
print(f'다른 스레드: {r3["messages"][-1].text}')

다른 스레드: 

저는 대화 세션이 종료되면 모든 내용을 초기화하며, 사용자님의 신원·개인정보·이전 대화 내역을 전혀 기억하거나 저장하지 않습니다. 따라서 현재로선 사용자님에 대해 알려진 것이 없습니다. 궁금한 점이 있거나 도움이 필요하시면 언제든지 말씀해 주세요!


### 2.1 SqliteSaver의 요점: 영구 저장

SqliteSaver와 같이, 영구 저장이 가능한 DB를 연결하면   
새로운 에이전트에서도 저장된 state를 연결할 수 있습니다.

In [8]:
# 현재 연결 종료 + 에이전트 삭제 (프로세스 재시작 시뮬레이션)
conn.close()
del agent
print('에이전트 삭제, DB 연결 종료')

# 저장 파일에 다시 연결하고 에이전트 생성
conn2 = sqlite3.connect(DB_PATH, check_same_thread=False)
checkpointer2 = SqliteSaver(conn2)

agent2 = create_agent(
    llm,
    tools=[],
    system_prompt="당신은 유능한 AI 어시스턴트입니다. 한국어로 간략하게 답변하세요.",
    checkpointer=checkpointer2,
)
print('새 에이전트 생성 (같은 DB 파일 연결)')

# 같은 thread_id로 저장된 대화 복원 확인
r4 = agent2.invoke(
    {"messages": [{"role": "user", "content": "아까 내가 뭐라고 했었지?"}]},
    config=config_t1
)
print(f'\n새 에이전트 응답: {r4["messages"][-1].text}')

에이전트 삭제, DB 연결 종료
새 에이전트 생성 (같은 DB 파일 연결)

새 에이전트 응답: 

처음에 **"Context Engineering(컨텍스트 엔지니어링)에 관심이 많다"**고 말씀하셨습니다. 이 분야의 특정 기법이나 실무 적용 방향 중 더 자세히 알아보고 싶은 부분이 있으면 알려주세요.


### 관찰 포인트

- 에이전트를 완전히 삭제(`del agent`)한 후 새로 만들었지만, 같은 DB 파일을 연결하면 저장된 대화를 복원합니다.
- 이것이 `SqliteSaver`의 장점으로, 서버 재시작이나 배포 후에도 대화 기록이 유지됩니다.
- 운영 환경에서는 `PostgresSaver`를 사용하면 여러 서버 인스턴스가 같은 대화 기록을 공유할 수 있습니다.

### 2.2 State History 조회

`get_state()`와 `get_state_history()`로 체크포인트된 상태를 조회할 수 있습니다.

In [9]:
# 현재 State 조회
snapshot = agent2.get_state(config_t1)
print(f'현재 메시지 수: {len(snapshot.values.get("messages", []))}')
print()

for i, msg in enumerate(snapshot.values.get('messages', [])):
    role = msg.type
    content = msg.text[:80] if msg.text else '(tool call)'
    print(f'  [{i}] {role}: {content}')

현재 메시지 수: 6

  [0] human: 나는 Context Engineering에 관심이 많다.
  [1] ai: 

Context Engineering은 AI(특히 대형 언어 모델)가 더 정확하고 일관된 결과를 생산하도록 `컨텍스트(정보·문맥·제약·도메인 
  [2] human: 내 관심사에 대해 알려줘.
  [3] ai: 

현재까지 공유하신 정보로는 **'컨텍스트 엔지니어링'**이 가장 명확한 관심사입니다. 이 분야를 깊이 다룰 경우 자연스럽게 다음 하위 영역에
  [4] human: 아까 내가 뭐라고 했었지?
  [5] ai: 

처음에 **"Context Engineering(컨텍스트 엔지니어링)에 관심이 많다"**고 말씀하셨습니다. 이 분야의 특정 기법이나 실무 적


In [11]:
# State 히스토리 조회 (각 스텝의 체크포인트)
history = list(agent2.get_state_history(config_t1))
print(f'히스토리 항목 수: {len(history)}')
for i, h in enumerate(history[:6]):
    msg_count = len(h.values.get('messages', []))
    print(f'  [{i}] checkpoint: {h.config["configurable"]["checkpoint_id"]}... ({msg_count}개 메시지)')

히스토리 항목 수: 9
  [0] checkpoint: 1f171098-422e-6ce4-8007-2d0049024994... (6개 메시지)
  [1] checkpoint: 1f171097-f558-67c7-8006-82a0b2b810b3... (5개 메시지)
  [2] checkpoint: 1f171097-f54f-6890-8005-09498facb719... (4개 메시지)
  [3] checkpoint: 1f171095-a0ce-6d26-8004-5bb71df841d4... (4개 메시지)
  [4] checkpoint: 1f171095-2bc3-69eb-8003-b7d545366432... (3개 메시지)
  [5] checkpoint: 1f171095-2bbc-6bf0-8002-9449e3590134... (2개 메시지)


In [12]:
# 정리: DB 연결 종료
conn2.close()

---
## 3. 장기 메모리: Store

체크포인팅은 에이전트가 처리하는 State나 Context를 저장하지만,   
세션이 바뀌거나 장기 작업이 지속되는 경우에는 별도의 Long-Term Memory가 필요합니다.   

Long-Term Memory에는 주로 다음과 같은 정보를 저장합니다.
- Semantic Memory: 사실과 개념 기억
- Episodic Memory: 과거 경험 기억
- Procedural Memory: 반복적인 작업 패턴 기억

<br><br>
### InMemoryStore

에이전트의 장기 기억을 담당하는 Store를 사용해 보겠습니다.
- Namespace (네임스페이스): 메모리를 폴더처럼 계층 구조로 관리
- Key (키): 각 메모리 항목의 고유 식별자
- Value (값): JSON 형태의 메모리 내용
- 모든 Thread에서 접근 가능 (Cross-thread)

```python
# 네임스페이스 구조 예시
store.put(("user_123", "preferences"), "lang", {"value": "korean"})
store.put(("user_123", "memories"), uuid, {"memory": "사용자는 AI 개발자"})
```

In [13]:
# InMemoryStore 기본 사용
store = InMemoryStore()

namespace = ("user_hong", "preferences")

store.put(namespace, "pref_lang", {"preference": "한국어를 선호합니다."})
store.put(namespace, "pref_style", {"preference": "간결한 답변을 좋아합니다."})
store.put(namespace, "pref_topic", {"preference": "AI와 프로그래밍에 관심이 많습니다."})

results = store.search(namespace)
print(f'저장된 메모리 수: {len(results)}')
for r in results:
    print(f'  [{r.key}] {r.value}')

item = store.get(namespace, "pref_lang")
print(f'\n직접 조회: {item.value}')

저장된 메모리 수: 3
  [pref_lang] {'preference': '한국어를 선호합니다.'}
  [pref_style] {'preference': '간결한 답변을 좋아합니다.'}
  [pref_topic] {'preference': 'AI와 프로그래밍에 관심이 많습니다.'}

직접 조회: {'preference': '한국어를 선호합니다.'}


### 3.1 Long-Term Memory 도구

도구 함수에서 `ToolRuntime`을 통해 Store에 접근할 수 있습니다.   
이를 통해 에이전트가 장기 메모리를 읽고 쓰는 도구를 사용할 수 있습니다.

In [14]:
@tool
def save_user_info(user_id: str, info: str, runtime: ToolRuntime) -> str:
    """사용자 정보를 장기 메모리에 저장합니다.
    user_id: 사용자 ID
    info: 저장할 정보 (문자열)"""
    s = runtime.store
    ns = ("users", user_id)
    memory_id = str(uuid.uuid4())
    s.put(ns, memory_id, {"memory": info})
    return f"\'{info}\' 정보가 user_id={user_id}에 저장되었습니다."

@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """사용자의 장기 메모리를 조회합니다.
    user_id: 사용자 ID"""
    s = runtime.store
    ns = ("users", user_id)
    items = s.search(ns)
    if not items:
        return f"user_id={user_id}에 저장된 정보가 없습니다."
    memories = [item.value["memory"] for item in items]
    return "\n".join(memories)

print('Store 도구 정의 완료')

Store 도구 정의 완료


In [21]:
# Store + Agent 생성 (Checkpointer도 SqliteSaver 사용)

mem_db_id = str(uuid.uuid4())[:8]

memory_store = InMemoryStore()
conn_mem = sqlite3.connect(f"memory_agent_{mem_db_id}.db", check_same_thread=False)
memory_checkpointer = SqliteSaver(conn_mem)

memory_agent = create_agent(
    llm,
    tools=[save_user_info, get_user_info],
    system_prompt="매번 도구를 사용하기 전, 한국어로 간략하게 설명하세요.",
    checkpointer=memory_checkpointer,
    store=memory_store,
)

print(f'Memory Agent 생성 완료 (memory_agent_{mem_db_id}.db)')

Memory Agent 생성 완료 (memory_agent_c769a86c.db)


In [22]:
# 세션 1: 사용자 정보 저장
config_s1 = {"configurable": {"thread_id": "session-1"}}

r_save = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "user_id가 hong인 사용자 정보를 저장해줘: '홍민성, 30세, 서울 거주, AI 개발자'"}]},
    config=config_s1
)
print(f'세션 1 (저장): {r_save["messages"][-1].text}')

세션 1 (저장): 

사용자 ID 'hong'에 대한 정보가 성공적으로 저장되었습니다.


In [23]:
r_save

{'messages': [HumanMessage(content="user_id가 hong인 사용자 정보를 저장해줘: '홍민성, 30세, 서울 거주, AI 개발자'", additional_kwargs={}, response_metadata={}, id='703d42c9-78e6-4c76-b3bf-5f6feae7c7ad'),
  AIMessage(content="\n\n사용자 ID가 'hong'인 사용자의 정보('홍민성, 30세, 서울 거주, AI 개발자')를 저장합니다.\n\n", additional_kwargs={'refusal': None, 'reasoning': 'The user wants to save information for a user with ID "hong".\nThe information to be saved is \'홍민성, 30세, 서울 거주, AI 개발자\'.\nI need to use the `save_user_info` tool.\nParameters:\n- user_id: "hong"\n- info: \'홍민성, 30세, 서울 거주, AI 개발자\'\n\nI will call the tool now.\nAfter calling the tool, I will confirm that the information has been saved.\nThe instruction requires me to briefly explain in Korean before using the tool.\n\nTool call description in Korean: 사용자 ID \'hong\'의 정보(\'홍민성, 30세, 서울 거주, AI 개발자\')를 저장합니다.\nThen execute the tool.\nThen reply to the user.\n\nWait, I should check if the user already has info? No, the request is just to save.\nSo, step 1: Explain action. 

In [24]:
# 세션 2: 다른 스레드에서 정보 조회 (Cross-thread)
config_s2 = {"configurable": {"thread_id": "session-2"}}

r_get = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "user_id가 hong인 사용자 정보 알려줘"}]},
    config=config_s2
)
print(f'세션 2 (조회): {r_get["messages"][-1].text}')

세션 2 (조회): 

hong라는 ID를 가진 사용자의 정보는 다음과 같습니다:

홍민성, 30세, 서울 거주, AI 개발자


In [25]:
r_get

{'messages': [HumanMessage(content='user_id가 hong인 사용자 정보 알려줘', additional_kwargs={}, response_metadata={}, id='0e93405e-1b9b-494e-ab02-6bf6f36c986f'),
  AIMessage(content='\n\nhong라는 ID를 가진 사용자의 장기 메모리 정보를 조회합니다.\n\n', additional_kwargs={'refusal': None, 'reasoning': 'The user is asking for the user info of a user with ID \'hong\'.\nI need to use the `get_user_info` function to retrieve this information.\nThe required parameter is `user_id`, which is provided as \'hong\'.\n\n1.  **Identify the goal**: Retrieve user info for user ID \'hong\'.\n2.  **Select the tool**: `get_user_info`.\n3.  **Prepare parameters**: `user_id` = "hong".\n4.  **Execute**: Call the tool.\n\nLet\'s construct the response in Korean as requested (briefly describing the action). I will call the tool now.cedural logic:\n-   Call `get_user_info(user_id="hong")`.\n-   Wait for result.\n-   Provide result to user.\n\nSince I can\'t wait for the result in this thought block, I will just generate the tool call. The us

In [26]:
# 정리
conn_mem.close()

## [실습] Categorical Memory 만들기   

아래의 3개 카테고리를 가진 메모리 구조를 구현하고,    
에이전트가 대화 내용을 바탕으로 카테고리별 정보를 저장하고 불러오는 도구를 구현하세요.

- semantic: 사실, 정체성, 선호 (영속적)         
  - "나는 AI 엔지니어야", "카페인 알러지 있어"    
- episodic: 과거에 있었던 일과 경험 (시점 있음)
  - "어제 강남에서 미팅했어", "지난주에 면접 봤어"    
- procedural: 반복 작업 절차와 작업 방식 선호 
  - "보고서는 항상 한국어로", "코드 답변은 짧은 코멘트만" 


In [36]:
# 도구 구현
@tool
def save_user_info(user_id: str, category:str, info: str, runtime: ToolRuntime) -> str:
    """사용자의 대화 내용에서, 기억할 내용을 추출하여 카테고리별 메모리에 저장합니다.
    user_id: 사용자 ID
    category: 카테고리 
    - semantic: 사실, 정체성, 선호 (영속적)         
      - "나는 AI 엔지니어야", "카페인 알러지 있어"    
    - episodic: 과거에 있었던 일과 경험 (시점 있음)
      - "어제 강남에서 미팅했어", "지난주에 면접 봤어"    
    - procedural: 반복 작업 절차와 작업 방식 선호 
      - "보고서는 항상 한국어로", "코드 답변은 짧은 코멘트만"
    info: 저장할 정보 (문자열)
    """
    if category not in ['semantic', 'episodic', 'procedural']:
        return "저장 오류: 카테고리는 반드시 semantic, episodic, procedural 중 하나여야 합니다."
    
    s = runtime.store
    ns = ("users", user_id, category)
    memory_id = str(uuid.uuid4())
    s.put(ns, memory_id, {"memory": info})
    return f"\'{info}\' 정보가 user_id={user_id}, category={category} 에 저장되었습니다."

@tool
def get_user_info(user_id: str, category:str, runtime: ToolRuntime) -> str:
    """사용자의 메모리에서 카테고리별 정보를 조회합니다.    
여러 분류가 필요한 경우 한 번에 하나씩 호출합니다.
    user_id: 사용자 ID
    category: 카테고리 
    - semantic: 사실, 정체성, 선호 (영속적)         
      - "나는 AI 엔지니어야", "카페인 알러지 있어"    
    - episodic: 과거에 있었던 일과 경험 (시점 있음)
      - "어제 강남에서 미팅했어", "지난주에 면접 봤어"    
    - procedural: 반복 작업 절차와 작업 방식 선호 
      - "보고서는 항상 한국어로", "코드 답변은 짧은 코멘트만"
    """
    if category not in ['semantic', 'episodic', 'procedural']:
        return "조회 오류: 카테고리는 반드시 semantic, episodic, procedural 중 하나여야 합니다."
    
    s = runtime.store
    ns = ("users", user_id, category)
    items = s.search(ns)
    if not items:
        return f"user_id={user_id}, category={category} 에 저장된 정보가 없습니다."
    memories = [item.value["memory"] for item in items]
    return "\n".join(memories)

print('Store 도구 정의 완료')


Store 도구 정의 완료


In [37]:
# 에이전트 구현
mem_db_id = str(uuid.uuid4())[:8]

memory_store = InMemoryStore()
conn_mem = sqlite3.connect(f"memory_agent_{mem_db_id}.db", check_same_thread=False)
memory_checkpointer = SqliteSaver(conn_mem)

memory_agent = create_agent(
    llm,
    tools=[save_user_info, get_user_info],
    system_prompt="매번 도구를 사용하기 전, 한국어로 간략하게 설명하세요. 개인화 메모리를 활용해 사용자를 기억하세요.",
    checkpointer=memory_checkpointer,
    store=memory_store,
)

print(f'Memory Agent 생성 완료 (memory_agent_{mem_db_id}.db)')

Memory Agent 생성 완료 (memory_agent_8849ff17.db)


In [38]:
# 실행
# 세션 1: 사용자 정보 저장
config_s1 = {"configurable": {"thread_id": "session-1"}}

r_save = memory_agent.invoke(
    {"messages": [{"role": "user", "content": """
사용자 ID: Dev1
안녕! 나는 AI 엔지니어야. 어제 강남에서 중요한 미팅이 있었는데, 
클라이언트 요구사항을 정리하는 데 하루 종일 걸렸어. 
근데 보고서를 항상 한국어로 작성하다 보니까 조금 피곤해져. 
아, 그리고 나는 카페인 알러지가 있어서 커피 대신 녹차를 좋아해. 
그래서 앞으로 코드 답변은 짧은 코멘트만 해주는 게 나을 것 같아.
"""}]},
    config=config_s1
)
print(f'세션 1 (저장): {r_save["messages"][-1].text}')
r_save["messages"]

세션 1 (저장): 

저는 AI 엔지니어인 Dev1 님의 정보를 다음과 같이 카테고리별로 기억했습니다.

**🧠 Semantic (사실 및 정체성)**
- AI 엔지니어입니다.
- 카페인 알러지가 있어 커피 대신 녹차를 좋아합니다.

**📅 Episodic (경험과 일화)**
- 어제 강남에서 중요한 클라이언트 미팅이 있었는데, 요구사항 정리로 하루를 보내셨네요. 고생 많으셨습니다!

**⚙️ Procedural (작업 방식 선호)**
- 보고서는 항상 **한국어**로 작성합니다.
- 코드 답변 시에는 **짧은 코멘트만** 포함합니다.

앞으로는 이러한 편들을 최대한 반영해 도와드리겠습니다. 오늘 나머지 일정도 힘내서 잘 마무리하시길 바랍니다!


[HumanMessage(content='\n사용자 ID: Dev1\n안녕! 나는 AI 엔지니어야. 어제 강남에서 중요한 미팅이 있었는데, \n클라이언트 요구사항을 정리하는 데 하루 종일 걸렸어. \n근데 보고서를 항상 한국어로 작성하다 보니까 조금 피곤해져. \n아, 그리고 나는 카페인 알러지가 있어서 커피 대신 녹차를 좋아해. \n그래서 앞으로 코드 답변은 짧은 코멘트만 해주는 게 나을 것 같아.\n', additional_kwargs={}, response_metadata={}, id='b3aaeb15-ca2e-40e9-a62e-3176c929d683'),
 AIMessage(content='\n\n안녕하세요! Dev1 님, 만나서 반갑습니다. 말씀해주신 이력과 작업 스타일을 바탕으로 기억해야 할 내용들을 메모리에 저장하겠습니다.\n\n', additional_kwargs={'refusal': None, 'reasoning': 'The user provided their User ID as "Dev1" and a message containing various pieces of information about themselves.\nI need to parse this information and save it into the memory system using the `save_user_info` tool.\n\nBreakdown of the user\'s message:\n1.  **Identity:** "나는 AI 엔지니어야." (I am an AI engineer.) -> Category: `semantic`, Info: "AI 엔지니어입니다."\n2.  **Past Experience:** "어제 강남에서 중요한 미팅이 있었는데, 클라이언트 요구사항을 정리하는 데 하루 종일 걸렸어." (Yesterday I had an important meeting in Gangnam, spent all day organizing client requireme

In [39]:
# 세션 2: 다른 스레드에서 정보 조회 (Cross-thread)
config_s2 = {"configurable": {"thread_id": "session-2"}}

r_get = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "안녕 Dev1이야. 오늘 음료 뭐 마실까?"}]},
    config=config_s2
)
print(f'세션 2 (조회): {r_get["messages"][-1].text}')

세션 2 (조회): 

Dev1님! 고객님은 카페인 알러지가 있으시다고 하셨죠? 따라서 커피는 피하시고, **녹차**를 강추합니다 😊 혹시 다른 음료가 필요하시면 언제든 말씀해 주세요!


---
## 4. 임베딩 기반 메모리 검색

`InMemoryStore`에 임베딩 모델을 설정하면, 벡터 유사도 기반 검색이 가능합니다.

임베딩은 채팅 모델과 별개의 모델입니다. 우리 vLLM은 채팅 전용이라 임베딩을 서빙하지 않으므로, 의미 검색에는 별도 임베딩 모델을 씁니다. 여기서는 OpenAIEmbeddings를 사용합니다.

In [40]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

indexed_store = InMemoryStore(
    index={
        "embed": embeddings,
        "dims": 1536,
        "fields": ["memory"]
    }
)

ns = ("hong", "memories")
indexed_store.put(ns, str(uuid.uuid4()), {"memory": "홍민성은 Python과 AI 개발을 좋아합니다."})
indexed_store.put(ns, str(uuid.uuid4()), {"memory": "홍민성은 서울 강남에 거주합니다."})
indexed_store.put(ns, str(uuid.uuid4()), {"memory": "홍민성은 매일 아침 조깅을 합니다."})
indexed_store.put(ns, str(uuid.uuid4()), {"memory": "홍민성은 커피보다 녹차를 선호합니다."})

results = indexed_store.search(ns, query="프로그래밍 관련", limit=2)
print('쿼리: "프로그래밍 관련"')
for r in results:
    print(f'  {r.value["memory"]}')

print()
results2 = indexed_store.search(ns, query="식음료 취향", limit=2)
print('쿼리: "식음료 취향"')
for r in results2:
    print(f'  {r.value["memory"]}')

쿼리: "프로그래밍 관련"
  홍민성은 Python과 AI 개발을 좋아합니다.
  홍민성은 매일 아침 조깅을 합니다.

쿼리: "식음료 취향"
  홍민성은 커피보다 녹차를 선호합니다.
  홍민성은 매일 아침 조깅을 합니다.


### 관찰 포인트

- `query="프로그래밍 관련"`을 주면 Python/AI 관련 메모리가 우선 반환됩니다.
- `query="식음료 취향"`을 주면 녹차 선호 메모리가 우선 반환됩니다.
- 단순 키워드 매칭이 아닌 의미적 유사도로 검색합니다.

---
## [실습] Reflection 기반 Episodic Memory + 임베딩 검색

  
이번 실습에서는 에이전트가 사용자 발화를 직접 확인하여,    
영속적으로 가치 있는 사실을 임베딩 인덱스에 저장하도록 구현해 보겠습니다.   

사용자의 입력으로부터, Store 검색을 통해 관련 있는 메모리를 시스템 프롬프트에 동적으로 추가합니다.

이를 해결하기 위해, 앞에서 만든 미들웨어를 연결합니다.

### 미들웨어 설계

`EpisodicMemoryMiddleware(AgentMiddleware)` 

| Hook | 시점 | 역할 |
|------|------|------|
| `wrap_model_call` / `awrap_model_call` | 매 LLM 호출 직전 | 마지막 사용자 발화로 Store 임베딩 검색을 수행해 관련 메모리만 시스템 프롬프트에 주입 |
| `after_agent` / `aafter_agent` | turn 종료 직후 | 사용자 발화에서 영속적 사실 0~N 개 추출해 Store 에 저장 |

힌트:
- Middleware 노트북의 `ContextInjectionMiddleware` 패턴을 활용하세요:
- 사실 추출은 오픈 모델로 수행합니다.
- Store 네임스페이스: `(user_id, "episodic_memories")`, key 는 `uuid4`, value 는 `{"memory": fact}`
- 검색은 `store.search(namespace, query=user_text, limit=top_k)`
- sync/async 짝을 함께 정의 (`stream_with_markdown` 호환)
- 일회성 질문("오늘 날씨 어때?")에서는 추출 결과가 빈 배열이어야 하므로, 이를 프롬프트에 명시하세요.

In [ ]:
# 빈 골격, 직접 채워보세요.
import json, uuid
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from langchain.messages import SystemMessage, HumanMessage

EXTRACT_PROMPT = """..."""  # TODO: 추출 규칙을 담은 프롬프트

class EpisodicMemoryMiddleware(AgentMiddleware):
    """사용자 발화에서 영속적 사실을 추출해 임베딩 Store 에 저장하고,
    매 LLM 호출 직전 관련 메모리만 검색해 시스템 프롬프트에 주입한다.

    - wrap_model_call: 마지막 사용자 발화로 임베딩 검색을 수행해 시스템 프롬프트에 주입
    - after_agent: turn 종료 시 사용자 발화에서 사실을 추출해 Store 에 저장
    - sync/async 짝(awrap_model_call/aafter_agent) 함께 정의 (stream_with_markdown 호환)"""

    def __init__(self, store, user_id, top_k=3, extractor=None):
        super().__init__()
        self.store = store
        self.user_id = user_id
        self.top_k = top_k
        self.namespace = (user_id, 'episodic_memories')
        self.extractor = extractor or llm

    def _last_human_text(self, messages):
        for m in reversed(messages):
            if isinstance(m, HumanMessage) or getattr(m, 'type', None) == 'human':
                return m.text
        return None

    def _parse_facts(self, raw):
        s = raw.strip()
        if s.startswith('```'):
            lines = s.splitlines()
            s = '\n'.join(lines[1:-1] if lines[-1].startswith('```') else lines[1:])
        try:
            facts = json.loads(s)
        except Exception:
            return []
        if not isinstance(facts, list):
            return []
        return [f.strip() for f in facts if isinstance(f, str) and f.strip()]

    def _retrieve(self, user_text):
        if not user_text:
            return []
        results = self.store.search(self.namespace, query=user_text, limit=self.top_k)
        return [r.value['memory'] for r in results]

    def _inject(self, request, memories):
        if not memories:
            return request
        block = '\n\n[사용자에 대해 알고 있는 정보]\n' + '\n'.join(f'- {m}' for m in memories)
        orig = request.system_message
        new_text = (orig.text + block) if orig else block
        print(f'  [Episodic] 관련 메모리 {len(memories)}개 주입')
        return request.override(system_message=SystemMessage(content=new_text))

    def _save_facts(self, facts):
        for f in facts:
            self.store.put(self.namespace, str(uuid.uuid4()), {'memory': f})
            print(f'  [Episodic] 새 사실 저장: {f}')


    def wrap_model_call(self, request, handler):
        # TODO: 사용자 발화로 메모리 검색을 수행하고 시스템 프롬프트에 주입한 뒤 handler 호출
        return handler(request)

    async def awrap_model_call(self, request, handler):
        return await handler(request)

    def after_agent(self, state, runtime):
        # TODO: 사용자 발화에서 사실을 추출해 Store 에 저장
        return None

    async def aafter_agent(self, state, runtime):
        return None


---
### 정답

In [41]:
import json
import uuid
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from langchain.messages import SystemMessage, HumanMessage


EXTRACT_PROMPT = """다음 사용자 발화에서, 사용자에 대해 새로 알게 된 영구적이고 재사용 가능한 사실을 0~3개 추출하세요.

규칙:
- 일회성 질문이나 그 자리에서만 의미 있는 발언은 무시 (예: "오늘 날씨 어때?")
- 사용자의 정체성/선호/상황/목표/이력/제약에 관련된 사실만 추출
- 각 사실은 한 문장, 주어는 "사용자"로 시작
- 새로 알게 된 게 없으면 빈 배열 []만 반환

사용자 발화: {user_text}

반드시 JSON 배열만 출력 (다른 설명이나 코드펜스 없이). 예: ["사용자는 ...", "사용자는 ..."]"""


class EpisodicMemoryMiddleware(AgentMiddleware):
    """사용자 발화에서 영속적 사실을 추출해 임베딩 Store 에 저장하고,
    매 LLM 호출 직전 관련 메모리만 검색해 시스템 프롬프트에 주입한다.

    - wrap_model_call: 마지막 사용자 발화로 임베딩 검색을 수행해 시스템 프롬프트에 주입
    - after_agent: turn 종료 시 사용자 발화에서 사실을 추출해 Store 에 저장
    - sync/async 짝(awrap_model_call/aafter_agent) 함께 정의 (stream_with_markdown 호환)"""

    def __init__(self, store, user_id, top_k=3, extractor=None):
        super().__init__()
        self.store = store
        self.user_id = user_id
        self.top_k = top_k
        self.namespace = (user_id, 'episodic_memories')
        self.extractor = extractor or llm

    def _last_human_text(self, messages):
        for m in reversed(messages):
            if isinstance(m, HumanMessage) or getattr(m, 'type', None) == 'human':
                return m.text
        return None

    def _parse_facts(self, raw):
        s = raw.strip()
        if s.startswith('```'):
            lines = s.splitlines()
            s = '\n'.join(lines[1:-1] if lines[-1].startswith('```') else lines[1:])
        try:
            facts = json.loads(s)
        except Exception:
            return []
        if not isinstance(facts, list):
            return []
        return [f.strip() for f in facts if isinstance(f, str) and f.strip()]

    def _retrieve(self, user_text):
        if not user_text:
            return []
        results = self.store.search(self.namespace, query=user_text, limit=self.top_k)
        return [r.value['memory'] for r in results]

    def _inject(self, request, memories):
        if not memories:
            return request
        block = '\n\n[사용자에 대해 알고 있는 정보]\n' + '\n'.join(f'- {m}' for m in memories)
        orig = request.system_message
        new_text = (orig.text + block) if orig else block
        print(f'  [Episodic] 관련 메모리 {len(memories)}개 주입')
        return request.override(system_message=SystemMessage(content=new_text))

    def _save_facts(self, facts):
        for f in facts:
            self.store.put(self.namespace, str(uuid.uuid4()), {'memory': f})
            print(f'  [Episodic] 새 사실 저장: {f}')

    def wrap_model_call(self, request, handler):
        user_text = self._last_human_text(request.messages)
        memories = self._retrieve(user_text)
        return handler(self._inject(request, memories))

    async def awrap_model_call(self, request, handler):
        user_text = self._last_human_text(request.messages)
        memories = self._retrieve(user_text)
        return await handler(self._inject(request, memories))

    def after_agent(self, state, runtime):
        user_text = self._last_human_text(state['messages'])
        if not user_text:
            return None
        raw = self.extractor.invoke(EXTRACT_PROMPT.format(user_text=user_text)).text
        self._save_facts(self._parse_facts(raw))
        return None

    async def aafter_agent(self, state, runtime):
        user_text = self._last_human_text(state['messages'])
        if not user_text:
            return None
        resp = await self.extractor.ainvoke(EXTRACT_PROMPT.format(user_text=user_text))
        self._save_facts(self._parse_facts(resp.text))
        return None

print('EpisodicMemoryMiddleware 정의 완료')

EpisodicMemoryMiddleware 정의 완료


In [42]:
# 임베딩 인덱스가 있는 Store + SqliteSaver + 에이전트 구성
ep_embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
ep_store = InMemoryStore(index={'embed': ep_embeddings, 'dims': 1536, 'fields': ['memory']})

ep_conn = sqlite3.connect('episodic.db', check_same_thread=False)
ep_checkpointer = SqliteSaver(ep_conn)

USER_ID = 'user1'

ep_agent = create_agent(
    llm,
    tools=[],
    system_prompt='당신은 유능한 AI 어시스턴트입니다. 한국어로 간결하게 답변하세요.',
    middleware=[EpisodicMemoryMiddleware(ep_store, USER_ID)],
    checkpointer=ep_checkpointer,
    store=ep_store,
)

print('Episodic Memory Agent 생성 완료')

Episodic Memory Agent 생성 완료


In [43]:
# Turn 1: 새 thread, 사용자가 자기 정보를 흘리면 사실 추출과 저장이 발생
cfg_a = {'configurable': {'thread_id': 'thread-A'}}

r1 = ep_agent.invoke(
    {'messages': [{'role': 'user',
                   'content': '안녕! 나는 김철수, 30살이고 서울 강남에 살아. AI 엔지니어로 일하고, 카페인 알러지가 있어서 커피는 못 마셔.'}]},
    config=cfg_a,
)
print(f'\n[응답] {r1["messages"][-1].text[:200]}')

print('\n현재 Store 의 사실 목록:')
for it in ep_store.search((USER_ID, 'episodic_memories')):
    print(f'  - {it.value["memory"]}')

  [Episodic] 새 사실 저장: 사용자는 이름이 김철수이고 30세이다.
  [Episodic] 새 사실 저장: 사용자는 서울 강남에 거주한다.
  [Episodic] 새 사실 저장: 사용자는 AI 엔지니어로 일하며 카페인을 섭취할 수 없다.

[응답] 

안녕하세요, 철수 님! 반갑습니다. AI 엔지니어로 일하시는 점, 강남 생활, 그리고 카페인 알러지 사항은 모두 잘 기록해두겠습니다. 궁금한 점이 있거나 도움이 필요하시면 언제든 말씀해 주세요. 건강 챙기며 활기찬 하루 보내세요!

현재 Store 의 사실 목록:
  - 사용자는 이름이 김철수이고 30세이다.
  - 사용자는 서울 강남에 거주한다.
  - 사용자는 AI 엔지니어로 일하며 카페인을 섭취할 수 없다.


In [44]:
# Turn 2: 다른 thread, 일회성 질문이라 새 사실은 추출되지 않아야 함
cfg_b = {'configurable': {'thread_id': 'thread-B'}}
before = len(ep_store.search((USER_ID, 'episodic_memories')))

r2 = ep_agent.invoke(
    {'messages': [{'role': 'user', 'content': '오늘 우리 동네 날씨 어때?'}]},
    config=cfg_b,
)
print(f'\n[응답] {r2["messages"][-1].text[:160]}')

after = len(ep_store.search((USER_ID, 'episodic_memories')))
print(f'\n사실 수 변화: {before} -> {after} (일회성 발화는 추출되지 않음)')

  [Episodic] 관련 메모리 3개 주입

[응답] 

실시간 날씨 데이터에 접근할 수 없어 정확한 강남 지역의 오늘 날씨를 안내해 드리기 어렵습니다. 기상청 날씨 누리나 스마트폰 기본 날씨 앱에서 바로 확인해 보시는 것을 권장합니다. 혹시 일정 계획에 맞춰 복장이나 외출 팁이 필요하면 언제든 말씀해 주세요.

사실 수 변화: 3 -> 3 (일회성 발화는 추출되지 않음)


In [45]:
# Turn 3: 또 다른 thread. 단기 메모리(checkpointer)에는 저장된 대화가 없지만
#         장기 메모리(Store)의 사실을 임베딩 검색해 시스템 프롬프트에 주입하므로 답변 가능
cfg_c = {'configurable': {'thread_id': 'thread-C'}}

r3 = ep_agent.invoke(
    {'messages': [{'role': 'user', 'content': '내가 무슨 일을 한다고 했지?'}]},
    config=cfg_c,
)
print(f'\n[응답] {r3["messages"][-1].text}')

  [Episodic] 관련 메모리 3개 주입

[응답] 

AI 엔지니어로 일하신다고 하셨습니다.


In [46]:
# Turn 4: 임베딩 검색으로 발화와 의미적으로 가까운 메모리만 끌려옴
#         "음료 추천" 발화를 하면 카페인 알러지 메모리가 우선 검색돼 카페인 없는 음료를 추천
cfg_d = {'configurable': {'thread_id': 'thread-D'}}

r4 = ep_agent.invoke(
    {'messages': [{'role': 'user', 'content': '카페에서 마실만한 음료 하나 추천해줘'}]},
    config=cfg_d,
)
print(f'\n[응답] {r4["messages"][-1].text}')

ep_conn.close()

  [Episodic] 관련 메모리 3개 주입

[응답] 

무카페인 **루이보스 차(Rooibos Tea)**를 추천합니다. 아예 카페인이 없는 남아공산 허브차로, 항산화 성분 풍부해 일의 피로 회복과 정신적 안정에 도움이 됩니다. 과실 시럽 추가나 우유 블렌드 등으로 취향에 따라 간단히 변형해서 즐기시면 좋습니다.


### 관찰 포인트

1. 자율적 메모리 형성    
   - 사용자가 요청하지 않아도, 별도의 LLM이 영속적 사실을 추출합니다.
2. Cross-thread 활용      
   - 단기 기억은 스레드에, 장기 기억은 스토어에 저장됩니다.
3. 선택적 컨텍스트 주입 (RAG over memory)    
   - 앞의 `save_user_info`/`get_user_info` 방식과 다르게,   
     발화와 의미적으로 가까운 `top_k`개만 시스템 프롬프트에 주입합니다. 
   - 메모리가 많아지더라도 컨텍스트가 크게 늘어나지는 않지만, 주요 메모리 누락 가능성이 있습니다.   
4. 미들웨어를 고려한 영속성 계층    
   - `wrap_model_call`(읽기) + `after_agent`(쓰기) 두 hook을 이용하여   
     최근의 에이전트들이 사용하는 ChatGPT/Claude 의 "Memory" 기능과 동일한 동작을 구현할 수 있습니다.

---
## 정리

이번 실습에서 배운 내용:

1. SqliteSaver: DB 파일에 체크포인트를 저장하여 프로세스 재시작 후에도 대화 복원
2. InMemoryStore: Cross-thread 장기 메모리로 사용자별 정보 영구 저장
3. ToolRuntime: 도구 함수에서 `runtime.store`로 Store에 접근하는 패턴
4. 임베딩 검색: 벡터 유사도 기반으로 관련 메모리를 효율적으로 검색

### 하네스 엔지니어링 관점

메모리는 하네스의 영속성 계층으로, 에이전트의 상태 관리를 담당합니다.

| 구성요소 | 하네스 역할 | 역할 |
|---------|-----------|--------|
| SqliteSaver | 단기 메모리 (Thread 내) | 대화 연속성, 장애 복구 |
| InMemoryStore | 장기 메모리 (Cross-thread) | 개인화, 학습 |
| 임베딩 검색 | 메모리 검색 최적화 | 관련 정보만 효율적 조회 |

## build_agent 익스텐션으로 메모리 배포

build_agent.py의 `EXTENSION_MODULES`에 memory를 더하면, 

`register()`가 돌려주는 EpisodicMemoryMiddleware가 배포 에이전트에 자동으로 적용됩니다. 

memory.py는 사실 추출 모델과 임베딩 Store를 만들어 미들웨어에 연결합니다.

아래에서 build_agent.py가 skills와 memory를 포함하는 memory.py를 만든 뒤,    
slack_app을 다시 시작하면 봇이 스킬과 메모리를 함께 사용합니다.

In [48]:
%%writefile build_agent.py
"""build_agent.py

LLM + Tool + MCP + Middleware + SKill + Memory Middleware

MCP 도구를 await로 받아오므로 build_agent는 async 함수입니다.
"""

from __future__ import annotations

import os

from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

from select_model import load_model
from tools import DEFAULT_TOOLS
from mcp_config import load_servers

DEFAULT_SYSTEM_PROMPT = (
    '당신은 유용한 AI 어시스턴트입니다. '
    '필요하면 도구를 사용하고, 도구를 실행하기 전 중간 과정을 간략히 설명하세요.'
)

# build_agent가 직접 연결하는 MCP 서버. mcp_servers.json의 stateless 서버를 읽는다.
# stateful 세션(Playwright 등)은 봇이 세션으로 열어 extra_tools로 넘긴다.
MCP_SERVERS = load_servers(stateful=False)

# 노트북 실습이 만든 배포용 익스텐션 모듈을 import해 미들웨어와 도구를 합친다.
# skills.py는 스킬 실습에서, memory.py는 메모리 실습에서 만든다. 파일이 없거나
# register()가 실패하면 그 모듈만 건너뛰고 나머지로 동작한다.
EXTENSION_MODULES = ['skills', 'memory']


def load_extensions():
    """EXTENSION_MODULES의 각 모듈을 import해 register()가 돌려주는 미들웨어와 도구를 모읍니다.

    각 모듈은 register()에서 {'middleware': [...], 'tools': [...]} 형태를 돌려줍니다.
    import 실패나 register() 오류는 그 모듈만 건너뛰고 메시지를 출력합니다.
    """
    import importlib

    middleware, tools = [], []
    for name in EXTENSION_MODULES:
        try:
            mod = importlib.import_module(name)
        except Exception as e:
            print(f'[build_agent] 익스텐션 {name}을 건너뜁니다(로드 실패): {type(e).__name__}: {e}')
            continue
        register = getattr(mod, 'register', None)
        if register is None:
            print(f'[build_agent] 익스텐션 {name}에 register()가 없어 건너뜁니다')
            continue
        try:
            spec = register() or {}
        except Exception as e:
            print(f'[build_agent] 익스텐션 {name}을 건너뜁니다(register 오류): {type(e).__name__}: {e}')
            continue
        mw = list(spec.get('middleware', []))
        tl = list(spec.get('tools', []))
        middleware += mw
        tools += tl
        print(f'[build_agent] 익스텐션 {name} 적용: 미들웨어 {len(mw)}개, 도구 {len(tl)}개')
    return middleware, tools


async def _auto_elicitation(mcp_context, params, context):
    """Slack MCP의 post_message가 채널을 되물을 때 기본 채널로 자동 응답합니다.

    봇이나 노트북에는 사용자에게 입력 폼을 표시할 방법이 없으므로, SLACK_CHANNEL_ID
    환경변수의 채널로 답합니다.
    """
    from mcp.types import ElicitResult

    channel = os.environ.get('SLACK_CHANNEL_ID', 'C_DRYRUN')
    return ElicitResult(action='accept', content={'channel_id': channel})


async def load_mcp_tools(servers=None):
    """MCP 서버에서 도구를 받아옵니다. 연결에 실패하면 빈 목록을 돌려줍니다."""
    servers = MCP_SERVERS if servers is None else servers
    if not servers:
        return []
    try:
        from langchain_mcp_adapters.callbacks import Callbacks

        client = MultiServerMCPClient(
            servers, callbacks=Callbacks(on_elicitation=_auto_elicitation)
        )
        return await client.get_tools()
    except Exception as e:
        print(f'[build_agent] MCP 도구 로드 실패, 표준 도구만 사용합니다: {e}')
        return []


async def build_agent(
    model_name=None,
    platform='openai',
    model=None,
    tools=None,
    extra_tools=None,
    middleware=None,
    system_prompt=None,
    checkpointer=None,
    mcp_servers=None,
    **model_kwargs,
):
    """현재까지 구성된 에이전트를 만들어 반환합니다.

    EXTENSION_MODULES에 등록된 익스텐션이 있으면 그 미들웨어와 도구를 자동으로 더합니다.

    Args:
        model_name: 모델 이름. select_model.load_model로 전달됩니다.
        platform: 'openai' | 'vllm' | 'ollama'. load_model로 전달됩니다.
        model: 이미 만든 chat model. 주면 model_name/platform은 무시됩니다.
        tools: 표준 도구 대신 쓸 도구 목록. 생략 시 tools.py의 DEFAULT_TOOLS.
        extra_tools: 호출자가 직접 만든 도구를 추가로 붙입니다. 봇이 stateful
            세션(예: Playwright)에서 받은 도구를 넘길 때 씁니다.
        middleware: create_agent에 넘길 미들웨어 목록. 노트북에서 만든
            미들웨어를 넣으면 그 동작이 배포 에이전트에 적용됩니다.
        system_prompt: 시스템 프롬프트. 생략 시 기본값을 씁니다.
        checkpointer: 멀티턴 대화를 위한 체크포인터.
        mcp_servers: 연결할 MCP 서버 설정. 생략 시 MCP_SERVERS, {}면 MCP를 끕니다.
        model_kwargs: load_model로 전달되는 추가 인자 (reasoning_effort 등).
    """
    if model is None:
        model = load_model(model_name, platform=platform, **model_kwargs)
    base_tools = list(DEFAULT_TOOLS) if tools is None else list(tools)
    mcp_tools = await load_mcp_tools(mcp_servers)
    extra = list(extra_tools) if extra_tools else []
    ext_middleware, ext_tools = load_extensions()
    if system_prompt is None:
        system_prompt = DEFAULT_SYSTEM_PROMPT
    return create_agent(
        model,
        tools=base_tools + mcp_tools + extra + ext_tools,
        middleware=(list(middleware) if middleware else []) + ext_middleware,
        system_prompt=system_prompt,
        checkpointer=checkpointer,
    )

Overwriting build_agent.py


In [49]:
%%writefile memory.py
"""memory.py

배포 에이전트에 에피소딕 메모리 기능을 더하는 익스텐션입니다. 사용자 발화에서
영속적 사실을 추출해 임베딩 Store에 저장하고, 매 모델 호출 직전 관련 메모리를 검색해
시스템 프롬프트에 주입합니다. build_agent가 register()를 호출해 미들웨어를 받습니다.
"""

import json
import uuid

from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from langchain.messages import SystemMessage, HumanMessage

# 배포 에이전트가 공유하는 사용자 식별자. InMemoryStore는 프로세스가 살아 있는 동안만 메모리를 유지한다.
USER_ID = 'slack_user'

# 미들웨어와 삭제 함수가 같은 Store를 보도록 모듈 단위로 한 번만 만든다.
_STORE = None


def get_store():
    """배포 에이전트가 공유하는 임베딩 Store를 돌려줍니다. 처음 호출할 때 한 번 만듭니다."""
    global _STORE
    if _STORE is None:
        from langchain_openai import OpenAIEmbeddings
        from langgraph.store.memory import InMemoryStore

        embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
        _STORE = InMemoryStore(index={'embed': embeddings, 'dims': 1536, 'fields': ['memory']})
    return _STORE


def clear_memory(user_id=USER_ID):
    """user_id의 에피소딕 메모리를 모두 지우고, 지운 개수를 돌려줍니다."""
    store = get_store()
    ns = (user_id, 'episodic_memories')
    items = store.search(ns)
    for it in items:
        store.delete(ns, it.key)
    return len(items)


EXTRACT_PROMPT = """다음 사용자 발화에서, 사용자에 대해 새로 알게 된 영구적이고 재사용 가능한 사실을 0~3개 추출하세요.

규칙:
- 일회성 질문이나 그 자리에서만 의미 있는 발언은 무시 (예: "오늘 날씨 어때?")
- 사용자의 정체성/선호/상황/목표/이력/제약에 관련된 사실만 추출
- 각 사실은 한 문장, 주어는 "사용자"로 시작
- 새로 알게 된 게 없으면 빈 배열 []만 반환

사용자 발화: {user_text}

반드시 JSON 배열만 출력 (다른 설명이나 코드펜스 없이). 예: ["사용자는 ...", "사용자는 ..."]"""


class EpisodicMemoryMiddleware(AgentMiddleware):
    """사용자 발화에서 영속적 사실을 추출해 임베딩 Store에 저장하고,
    매 LLM 호출 직전 관련 메모리만 검색해 시스템 프롬프트에 주입한다.

    - wrap_model_call: 마지막 사용자 발화로 임베딩 검색을 수행해 시스템 프롬프트에 주입
    - after_agent: turn 종료 시 사용자 발화에서 사실을 추출해 Store에 저장
    - sync/async 짝(awrap_model_call/aafter_agent)을 함께 정의
    """

    def __init__(self, store, user_id, top_k=3, extractor=None):
        super().__init__()
        self.store = store
        self.user_id = user_id
        self.top_k = top_k
        self.namespace = (user_id, 'episodic_memories')
        self.extractor = extractor

    def _last_human_text(self, messages):
        for m in reversed(messages):
            if isinstance(m, HumanMessage) or getattr(m, 'type', None) == 'human':
                return m.text
        return None

    def _parse_facts(self, raw):
        s = raw.strip()
        if s.startswith('```'):
            lines = s.splitlines()
            s = '\n'.join(lines[1:-1] if lines[-1].startswith('```') else lines[1:])
        try:
            facts = json.loads(s)
        except Exception:
            return []
        if not isinstance(facts, list):
            return []
        return [f.strip() for f in facts if isinstance(f, str) and f.strip()]

    def _retrieve(self, user_text):
        if not user_text:
            return []
        results = self.store.search(self.namespace, query=user_text, limit=self.top_k)
        return [r.value['memory'] for r in results]

    def _inject(self, request, memories):
        if not memories:
            return request
        block = '\n\n[사용자에 대해 알고 있는 정보]\n' + '\n'.join(f'- {m}' for m in memories)
        orig = request.system_message
        new_text = (orig.text + block) if orig else block
        print(f'  [Episodic] 관련 메모리 {len(memories)}개 주입')
        return request.override(system_message=SystemMessage(content=new_text))

    def _save_facts(self, facts):
        for f in facts:
            self.store.put(self.namespace, str(uuid.uuid4()), {'memory': f})
            print(f'  [Episodic] 새 사실 저장: {f}')

    def wrap_model_call(self, request, handler):
        user_text = self._last_human_text(request.messages)
        memories = self._retrieve(user_text)
        return handler(self._inject(request, memories))

    async def awrap_model_call(self, request, handler):
        user_text = self._last_human_text(request.messages)
        memories = self._retrieve(user_text)
        return await handler(self._inject(request, memories))

    def after_agent(self, state, runtime):
        user_text = self._last_human_text(state['messages'])
        if not user_text or self.extractor is None:
            return None
        raw = self.extractor.invoke(EXTRACT_PROMPT.format(user_text=user_text)).text
        self._save_facts(self._parse_facts(raw))
        return None

    async def aafter_agent(self, state, runtime):
        user_text = self._last_human_text(state['messages'])
        if not user_text or self.extractor is None:
            return None
        resp = await self.extractor.ainvoke(EXTRACT_PROMPT.format(user_text=user_text))
        self._save_facts(self._parse_facts(resp.text))
        return None


def register():
    """build_agent가 호출하는 진입점입니다. 에피소딕 메모리 미들웨어를 돌려줍니다.

    공유 Store와 사실 추출 모델을 미들웨어에 연결합니다. 추출은 OpenAI 모델로 수행합니다.
    """
    from select_model import load_model

    extractor = load_model(platform='openai')
    return {
        'middleware': [EpisodicMemoryMiddleware(get_store(), USER_ID, extractor=extractor)],
        'tools': [],
    }

Writing memory.py


In [50]:
# memory.py를 만든 뒤 build_agent를 다시 import합니다.
import importlib, build_agent as build_agent_module
importlib.reload(build_agent_module)
from build_agent import build_agent

# middleware를 직접 넘기지 않아도 build_agent가 skills.py와 memory.py를 자동으로 싣습니다.
agent = await build_agent(model=llm, mcp_servers={})

result = await agent.ainvoke({
    'messages': [{'role': 'user', 'content': '나는 녹차를 좋아하고 커피는 안 마셔. 기억해줘.'}]
})
print(result['messages'][-1].text)

[build_agent] 익스텐션 skills 적용: 미들웨어 1개, 도구 4개
[build_agent] 익스텐션 memory 적용: 미들웨어 1개, 도구 0개
  [Episodic] 새 사실 저장: 사용자는 녹차를 좋아한다.
  [Episodic] 새 사실 저장: 사용자는 커피를 마시지 않는다.


네, 녹차를 좋아하고 커피는 드시지 않는 취향을 잘 memorize 했습니다. 필요하실 때 참고하여 추천이나 답변을 드리겠습니다.


## [실습] 메모리 삭제

발화와 의미가 가까운 사실만 골라 지우는 forget 도구를 만들어 
memory.py register의 tools에 추가하세요.    
`store.search(ns, query=...)`로 후보를 찾고 `store.delete(ns, key)`로 지웁니다.


참고: slack_app.py에는 clear_memory를 호출하는 핸들러가 있습니다.     
채널에서 봇을 멘션하고 "메모리 초기화"를 입력하여 메모리를 완전히 비울 수 있습니다.

In [ ]:
from memory import get_store, clear_memory, USER_ID

ns = (USER_ID, 'episodic_memories')
print('지우기 전 사실 수:', len(get_store().search(ns)))

removed = clear_memory()
print(f'{removed}개 삭제')
print('지운 뒤 사실 수:', len(get_store().search(ns)))